# simtwo two-node entanglement distribution

In [ ]:
from __future__ import annotations

import sys
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from matplotlib import pyplot as plt

from sequence.constants import MILLISECOND
from sequence.entanglement_management.generation import EntanglementGenerationA
from sequence.kernel.event import Event
from sequence.kernel.process import Process
from sequence.kernel.timeline import Timeline
from sequence.resource_management.rule_manager import Rule
from sequence.topology.node import QuantumRouter, BSMNode

In [ ]:
# running this in a notebook outside of the virtual env, so this code is only strictly necessary for importing simtwo:
# (otherwise, the next set of imports won't work)
# there might be a better way to do this, but this works for me just for demo purposes
cwd = Path.cwd()
for p in [cwd, cwd.parent, cwd / "src", cwd.parent / "src"]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

In [ ]:
from simtwo.core.ThermalClassicalChannel import ThermalClassicalChannel
from simtwo.core.ThermalQuantumChannel import ThermalQuantumChannel
from simtwo.core.sequence.link_model_manager import LinkModelManager

In [ ]:
csv_path = "ed_test_filtered_200.csv"

df = pd.read_csv(csv_path)
df = df.dropna(subset=["temperature_x"]).reset_index(drop=True)
df[["temperature_x"]].head()

In [ ]:
def eg_rule_condition(memory_info, manager, args):
    if memory_info.state == "RAW":
        return [memory_info]
    else:
        return []


def eg_rule_action1(memories_info, args):
    def eg_req_func(protocols, args):
        for protocol in protocols:
            if isinstance(protocol, EntanglementGenerationA):
                return protocol

    memories = [info.memory for info in memories_info]
    memory = memories[0]
    protocol = EntanglementGenerationA.create(None, "EGA." + memory.name, "m1", "r2", memory)
    protocol.primary = True

    return [protocol, ["r2"], [eg_req_func], [None]]


def eg_rule_action2(memories_info, args):
    memories = [info.memory for info in memories_info]
    memory = memories[0]
    protocol = EntanglementGenerationA.create(None, "EGA." + memory.name, "m1", "r1", memory)

    return [protocol, [None], [None], [None]]


In [ ]:
@dataclass
class TemperatureCsvUpdater:
    timeline: Timeline
    link_model_manager: LinkModelManager
    rows: list[dict]
    update_period_ps: int
    index: int = 0

    def start(self):
        self._schedule_update(0)

    def _schedule_update(self, time_ps):
        process = Process(self, "update", [], {})
        self.timeline.schedule(Event(int(time_ps), process))

    def update(self):
        if self.index >= len(self.rows):
            return

        self.link_model_manager.apply_to_registered_links(self.rows[self.index])
        self.index += 1

        next_time = self.timeline.now() + self.update_period_ps
        if self.index < len(self.rows) and next_time <= self.timeline.stop_time:
            self._schedule_update(next_time)


def force_light_speed(channel, light_speed_m_per_ps):
    if hasattr(channel, "light_speed"):
        channel.light_speed = float(light_speed_m_per_ps)

    if hasattr(channel, "_refresh_channel_params"):
        channel._refresh_channel_params()


In [ ]:
def test():
    sim_time = 1000
    qc_atten = 1e-4

    ps_per_ms = 1e9

    # to match the other experiment:
    full_path_distance_m = 32_000
    arm_distance_m = full_path_distance_m / 2

    alpha_per_c = 5e-7
    
    # temp start temp:
    t0_c = 20.0
    light_speed_m_per_ps = 0.0002
    qchannel_frequency = 1e12

    sim_time_ps = int(sim_time * ps_per_ms)
    update_period_ps = int(1 * ps_per_ms)

    temperature_rows = [
        {"temperature_x": float(temp)}
        for temp in df["temperature_x"].iloc[: sim_time + 1]
    ]

    tl = Timeline(sim_time_ps)

    r1 = QuantumRouter("r1", tl, 50)
    r2 = QuantumRouter("r2", tl, 50)
    m1 = BSMNode("m1", tl, ["r1", "r2"])

    r1.set_seed(0)
    r2.set_seed(1)
    m1.set_seed(2)

    for node in [r1, r2]:
        memory_array = node.get_components_by_type("MemoryArray")[0]
        # memory_array.update_memory_params("coherence_time", 0.3)

    nodes = [r1, r2, m1]
    classical_channels = {}

    for node1 in nodes:
        for node2 in nodes:
            if node1 == node2:
                continue

            if {node1.name, node2.name} == {"r1", "r2"}:
                base_distance_m = full_path_distance_m
            else:
                base_distance_m = arm_distance_m

            cc = ThermalClassicalChannel(
                "_".join(["cc", node1.name, node2.name]),
                tl,
                base_distance_m=base_distance_m,
                alpha_per_C=alpha_per_c,
                T0_C=t0_c,
            )
            force_light_speed(cc, light_speed_m_per_ps)
            cc.set_ends(node1, node2.name)
            classical_channels[(node1.name, node2.name)] = cc

    qc1 = ThermalQuantumChannel(
        "qc_r1_m1",
        tl,
        base_distance_m=arm_distance_m,
        alpha_per_C=alpha_per_c,
        T0_C=t0_c,
        attenuation=qc_atten,
        light_speed=light_speed_m_per_ps,
        frequency=qchannel_frequency,
    )
    qc1.set_ends(r1, m1.name)

    qc2 = ThermalQuantumChannel(
        "qc_r2_m1",
        tl,
        base_distance_m=arm_distance_m,
        alpha_per_C=alpha_per_c,
        T0_C=t0_c,
        attenuation=qc_atten,
        light_speed=light_speed_m_per_ps,
        frequency=qchannel_frequency,
    )
    qc2.set_ends(r2, m1.name)

    link_model_manager = LinkModelManager(
        base_distance_m=full_path_distance_m,
        alpha_per_c=alpha_per_c,
        t0_c=t0_c,
        light_speed_m_per_ps=light_speed_m_per_ps,
    )

    link_model_manager.register_group(
        "r1_to_m1_quantum_arm",
        [qc1],
        delay_fraction=0.5,
        distance_fraction=0.5,
    )

    link_model_manager.register_group(
        "r2_to_m1_quantum_arm",
        [qc2],
        delay_fraction=0.5,
        distance_fraction=0.5,
    )

    full_classical = [
        channel
        for (src, dst), channel in classical_channels.items()
        if {src, dst} == {"r1", "r2"}
    ]

    arm_classical = [
        channel
        for (src, dst), channel in classical_channels.items()
        if {src, dst} != {"r1", "r2"}
    ]

    link_model_manager.register_group(
        "r1_r2_classical_full_path",
        full_classical,
        delay_fraction=1.0,
        distance_fraction=1.0,
    )

    link_model_manager.register_group(
        "router_bsm_classical_arms",
        arm_classical,
        delay_fraction=0.5,
        distance_fraction=0.5,
    )

    link_model_manager.apply_to_registered_links(temperature_rows[0])

    updater = TemperatureCsvUpdater(
        timeline=tl,
        link_model_manager=link_model_manager,
        rows=temperature_rows,
        update_period_ps=update_period_ps,
    )
    updater.start()

    tl.init()

    rule1 = Rule(10, eg_rule_action1, eg_rule_condition, None, None)
    r1.resource_manager.load(rule1)

    rule2 = Rule(10, eg_rule_action2, eg_rule_condition, None, None)
    r2.resource_manager.load(rule2)

    tick = time.time()
    tl.run()
    print("execution time %.2f sec" % (time.time() - tick))

    data = []
    for info in r1.resource_manager.memory_manager:
        if info.entangle_time > 0:
            data.append(info.entangle_time / MILLISECOND)

    data.sort()

    plt.plot(data, range(1, len(data) + 1), marker="o")
    plt.xlabel("Simulation Time (ms)")
    plt.ylabel("Aggregated Number of Entangled Memory")
    plt.show()

    print(f"Entangled time data = {data}")

    return data


In [ ]:
data = test()
